In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

n_users = encoders["n_users"]
n_movies = encoders["n_movies"]

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

global_mean = train_df["rating"].mean()

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (802553, 6), Test: (197656, 6)


In [2]:
train_matrix = csr_matrix(
    (train_df["rating"], (train_df["user_idx"], train_df["movie_idx"])),
    shape=(n_users, n_movies)
)

# Item-item similarity: similarity between movie columns
# transpose so rows = movies, cols = users
item_similarity = cosine_similarity(train_matrix.T, dense_output=False)

print(f"Item similarity matrix shape: {item_similarity.shape}")
print(f"Type: {type(item_similarity)}")

Item similarity matrix shape: (3706, 3706)
Type: <class 'scipy.sparse._csr.csr_matrix'>


In [3]:
# Build a fast lookup: user_idx -> {movie_idx: rating} from train
from collections import defaultdict

user_ratings_dict = defaultdict(dict)
for row in train_df.itertuples(index=False):
    user_ratings_dict[row.user_idx][row.movie_idx] = row.rating

item_similarity_csr = item_similarity.tocsr()

def predict_rating_item_cf(user_idx, movie_idx, k=20):
    """Predict rating using top-k similar movies the user has rated."""
    user_rated = user_ratings_dict.get(user_idx, {})
    if not user_rated:
        return global_mean

    # Get similarities of target movie to all movies
    sim_row = item_similarity_csr[movie_idx]
    sim_dict = dict(zip(sim_row.indices, sim_row.data))

    # Intersect with movies this user has rated
    sims_and_ratings = [
        (sim_dict[m], r) for m, r in user_rated.items() if m in sim_dict and sim_dict[m] > 0
    ]

    if not sims_and_ratings:
        return global_mean

    # Take top-k by similarity
    sims_and_ratings.sort(key=lambda x: -x[0])
    sims_and_ratings = sims_and_ratings[:k]

    sims = np.array([s for s, r in sims_and_ratings])
    rats = np.array([r for s, r in sims_and_ratings])

    if sims.sum() == 0:
        return global_mean

    return np.dot(sims, rats) / sims.sum()

# Test on a small sample first (full test set would be slow in pure Python loop)
sample_test = test_df.sample(2000, random_state=42)

preds = [
    predict_rating_item_cf(row.user_idx, row.movie_idx)
    for row in sample_test.itertuples(index=False)
]

sample_test = sample_test.copy()
sample_test["pred_item_cf"] = preds

rmse_item_cf = rmse(sample_test["rating"], sample_test["pred_item_cf"])
mae_item_cf = mae(sample_test["rating"], sample_test["pred_item_cf"])

print(f"Item-Item CF (sample of 2000) -> RMSE: {rmse_item_cf:.4f}, MAE: {mae_item_cf:.4f}")

Item-Item CF (sample of 2000) -> RMSE: 1.0213, MAE: 0.7941


In [4]:
sample_test_large = test_df.sample(10000, random_state=42)

preds_large = [
    predict_rating_item_cf(row.user_idx, row.movie_idx)
    for row in sample_test_large.itertuples(index=False)
]

sample_test_large = sample_test_large.copy()
sample_test_large["pred_item_cf"] = preds_large

rmse_item_cf_large = rmse(sample_test_large["rating"], sample_test_large["pred_item_cf"])
mae_item_cf_large = mae(sample_test_large["rating"], sample_test_large["pred_item_cf"])

print(f"Item-Item CF (sample of 10000) -> RMSE: {rmse_item_cf_large:.4f}, MAE: {mae_item_cf_large:.4f}")

Item-Item CF (sample of 10000) -> RMSE: 0.9937, MAE: 0.7693


In [5]:
import pickle

MODELS_DIR = PROJECT_ROOT / "models"

with open(MODELS_DIR / "item_similarity.pkl", "wb") as f:
    pickle.dump(item_similarity_csr, f)

with open(MODELS_DIR / "item_cf_results.pkl", "wb") as f:
    pickle.dump({
        "rmse": rmse_item_cf_large,
        "mae": mae_item_cf_large,
        "sample_size": 10000,
    }, f)

print("Saved item_similarity.pkl and item_cf_results.pkl to models/")

Saved item_similarity.pkl and item_cf_results.pkl to models/


In [6]:
from scipy.sparse.linalg import svds

# Mean-center the ratings per user (helps SVD performance)
user_means = train_df.groupby("user_idx")["rating"].mean()

train_matrix_dense_centered = train_matrix.toarray().astype(float)
for u_idx, mean_r in user_means.items():
    row = train_matrix_dense_centered[u_idx]
    row[row != 0] -= mean_r

# Run SVD with k=50 latent factors
k = 50
U, sigma, Vt = svds(train_matrix_dense_centered, k=k)
sigma_diag = np.diag(sigma)

print(f"U: {U.shape}, sigma: {sigma_diag.shape}, Vt: {Vt.shape}")

U: (6040, 50), sigma: (50, 50), Vt: (50, 3706)


In [7]:
# Reconstruct the ratings matrix from latent factors
predicted_matrix = U @ sigma_diag @ Vt

# Add back user means (since we mean-centered before SVD)
for u_idx, mean_r in user_means.items():
    predicted_matrix[u_idx] += mean_r

# Predict on test set
test_df = test_df.copy()
test_df["pred_svd"] = test_df.apply(
    lambda row: predicted_matrix[row["user_idx"], row["movie_idx"]], axis=1
)

# Clip predictions to valid rating range [1, 5]
test_df["pred_svd"] = test_df["pred_svd"].clip(1, 5)

rmse_svd = rmse(test_df["rating"], test_df["pred_svd"])
mae_svd = mae(test_df["rating"], test_df["pred_svd"])

print(f"SVD (k={k}) -> RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f}")

SVD (k=50) -> RMSE: 1.0287, MAE: 0.8076


In [8]:
with open(MODELS_DIR / "svd_results.pkl", "wb") as f:
    pickle.dump({
        "k": k,
        "rmse": rmse_svd,
        "mae": mae_svd,
    }, f)

print("Saved svd_results.pkl")

Saved svd_results.pkl


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

movies_sdf_path = PROJECT_ROOT / "data" / "raw" / "movies.dat"
movies = pd.read_csv(
    movies_sdf_path, sep="::", engine="python",
    names=["movie_id", "title", "genres"], encoding="ISO-8859-1"
)

# Map movie_id -> movie_idx using the same encoder
with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)
movie_to_idx = encoders["movie_to_idx"]

movies["movie_idx"] = movies["movie_id"].map(movie_to_idx)
movies = movies.dropna(subset=["movie_idx"]).copy()
movies["movie_idx"] = movies["movie_idx"].astype(int)
movies = movies.sort_values("movie_idx").reset_index(drop=True)

# TF-IDF on genres (treat genre list as a "document")
movies["genres_str"] = movies["genres"].str.replace("|", " ", regex=False)
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genres_str"])

print(f"Genre TF-IDF matrix shape: {genre_matrix.shape}")
print(f"Vocabulary: {tfidf.get_feature_names_out()}")

Genre TF-IDF matrix shape: (3706, 20)
Vocabulary: ['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'horror' 'musical' 'mystery'
 'noir' 'romance' 'sci' 'thriller' 'war' 'western']


In [10]:
genre_similarity = cosine_similarity(genre_matrix, dense_output=False)
genre_similarity_csr = genre_similarity.tocsr()

def recommend_similar_by_content(movie_idx, top_n=5):
    sim_row = genre_similarity_csr[movie_idx]
    sim_dict = dict(zip(sim_row.indices, sim_row.data))
    sim_dict.pop(movie_idx, None)  # remove itself
    top_similar = sorted(sim_dict.items(), key=lambda x: -x[1])[:top_n]
    result_idx = [idx for idx, _ in top_similar]
    return movies[movies["movie_idx"].isin(result_idx)][["title", "genres"]]

# Example: Toy Story
example_movie = movies[movies["title"].str.contains("Toy Story", case=False)].iloc[0]
print(f"Movies similar to: {example_movie['title']} ({example_movie['genres']})\n")
recommend_similar_by_content(example_movie["movie_idx"])

Movies similar to: Toy Story (1995) (Animation|Children's|Comedy)



,title,genres
2162,"Bug's Life, A (1998)",Animation|Children's|Comedy
2898,Toy Story 2 (1999),Animation|Children's|Comedy
3371,Saludos Amigos (1943),Animation|Children's|Comedy
3508,Chicken Run (2000),Animation|Children's|Comedy
3511,"Adventures of Rocky and Bullwinkle, The (2000)",Animation|Children's|Comedy


In [11]:
with open(MODELS_DIR / "genre_similarity.pkl", "wb") as f:
    pickle.dump(genre_similarity_csr, f)

movies.to_parquet(PROCESSED_DIR / "movies_with_idx.parquet", index=False)

print("Saved genre_similarity.pkl and movies_with_idx.parquet")

Saved genre_similarity.pkl and movies_with_idx.parquet
